# Away Runs Boxplots by Weather Bracket for Every Team

This notebook creates one figure per team showing the spread of `away_runs_scored` across:

- temperature brackets
- humidity brackets
- pressure brackets

Each figure is saved to `analysis/boxplots/[TEAM]_boxplot.png`, where `[TEAM]` matches the team abbreviations already used elsewhere in the project.

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)

TEAM_PARAMS_PATH = 'team_parameters.csv'
LEAGUE_DATA_PATH = os.path.join('..', 'data', 'league_weather_2021_2025.csv')
OUTPUT_DIR = 'boxplots'

params_df = pd.read_csv(TEAM_PARAMS_PATH)
league_data = pd.read_csv(LEAGUE_DATA_PATH)
league_data['game_date'] = pd.to_datetime(league_data['game_date'])

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Teams loaded: {len(params_df)}')
print(f'League rows loaded: {len(league_data)}')
print(f'Output directory: {os.path.abspath(OUTPUT_DIR)}')

In [ ]:
def format_interval_labels(categories):
    return [str(cat).replace('(', '').replace(']', '').replace(', ', ' to ') for cat in categories]


def add_weather_bins(df):
    df = df.copy()
    df['temp_bin'] = pd.qcut(df['temp_f'], q=5, duplicates='drop')
    df['rhum_bin'] = pd.qcut(df['rhum'], q=5, duplicates='drop')
    df['pres_bin'] = pd.qcut(df['pres'], q=5, duplicates='drop')
    return df


def draw_boxplot_panel(ax, df, bin_col, title, color):
    categories = df[bin_col].cat.categories
    labels = format_interval_labels(categories)
    box_data = [df.loc[df[bin_col] == category, 'away_runs_scored'].dropna().values for category in categories]

    bp = ax.boxplot(
        box_data,
        patch_artist=True,
        labels=labels,
        medianprops=dict(color='black', linewidth=1.5),
    )

    for patch in bp['boxes']:
        patch.set(facecolor=color, alpha=0.55)

    ax.set_title(title)
    ax.tick_params(axis='x', rotation=20)
    ax.grid(axis='y', alpha=0.3)


def make_team_boxplot(team_code, team_name, stadium_name, team_df, output_dir, show=False):
    team_df = add_weather_bins(team_df)

    fig, axes = plt.subplots(1, 3, figsize=(22, 7), sharey=True)

    draw_boxplot_panel(axes[0], team_df, 'temp_bin', 'Away Runs by Temperature Bracket', '#d62728')
    draw_boxplot_panel(axes[1], team_df, 'rhum_bin', 'Away Runs by Humidity Bracket', '#1f77b4')
    draw_boxplot_panel(axes[2], team_df, 'pres_bin', 'Away Runs by Pressure Bracket', '#9467bd')

    axes[0].set_ylabel('Away Runs Scored')
    axes[0].set_xlabel('Temperature Bracket')
    axes[1].set_xlabel('Humidity Bracket')
    axes[2].set_xlabel('Pressure Bracket')

    fig.suptitle(f'{stadium_name} ({team_name}) — Away Runs Spread Across Weather Brackets', fontsize=15, y=1.02)
    plt.tight_layout()

    output_path = os.path.join(output_dir, f'{team_code}_boxplot.png')
    fig.savefig(output_path, dpi=150, bbox_inches='tight', facecolor='white')
    if show:
        plt.show()
    else:
        plt.close(fig)
    return output_path

In [ ]:
results = []

for _, row in params_df.iterrows():
    team_code = row['team_code']
    team_name = row['team_name']
    stadium_name = row['stadium_name']

    team_df = league_data[league_data['home_team'] == team_code].copy()
    if len(team_df) == 0:
        print(f'Skipping {team_code}: no league weather rows found')
        continue

    output_path = make_team_boxplot(team_code, team_name, stadium_name, team_df, OUTPUT_DIR, show=False)
    results.append({
        'team_code': team_code,
        'team_name': team_name,
        'stadium_name': stadium_name,
        'games': len(team_df),
        'output_file': os.path.basename(output_path),
    })
    print(f'Saved {team_code}: {output_path}')

results_df = pd.DataFrame(results)
results_df

In [ ]:
summary_path = os.path.join(OUTPUT_DIR, 'boxplot_generation_summary.csv')
results_df.to_csv(summary_path, index=False)
print(f'Saved summary: {os.path.abspath(summary_path)}')